# ADKP STRIDES report

Month-over-month data volume and egress metrics for the ADKP project (`project_id = syn2580853`).

Sections:
1. Data volume — latest version of files (what downloads default to)
2. Data volume — all versions across the project lifecycle
3. Egress by billing source, month-over-month
4. Egress by study (STRIDES > 10 TiB/month) — migration candidates

In [ ]:
from datetime import datetime, timezone
from IPython.display import Markdown, display

# Execution timestamp — evaluated when the notebook runs, so the report is self-dating.
_generated_at = datetime.now(timezone.utc).astimezone()
display(Markdown(f"*Report generated: {_generated_at:%Y-%m-%d %H:%M %Z}*"))

## Setup

Runs against Snowflake via `snowflake-connector-python` and returns pandas DataFrames. Run cells top-to-bottom, then run this command to produce a one-off PDF of the results.

```
python -m nbconvert --to html --execute \
  --ExecutePreprocessor.timeout=600 \
  --TagRemovePreprocessor.enabled=True \
  --TagRemovePreprocessor.remove_cell_tags remove-cell \
  --TagRemovePreprocessor.remove_input_tags remove-input \
  --TagRemovePreprocessor.remove_all_outputs_tags remove-output \
  adkp_strides_report.ipynb
```

Requires `snowflake-connector-python`, `pandas`, and `matplotlib` in the kernel VSCode is using:

```bash
pip install snowflake-connector-python pandas matplotlib
```

Connects by name to the `dataanalytics` connection. Authentication (OAuth/SSO) is whatever that connection block in `~/.snowflake/config.toml` defines — running this cell triggers the OAuth/browser flow automatically; nothing is hardcoded here.

The report reads `SYNAPSE_DATA_WAREHOUSE`; if the connection's own role/warehouse can't query it, set `ROLE` / `WAREHOUSE` below to ones that can.

In [ ]:
import os
import contextlib
import pandas as pd
import matplotlib.pyplot as plt
import snowflake.connector

# Connection defined in ~/.snowflake/config.toml (see `snow connection list`).
CONNECTION_NAME = "dataanalytics"

# Optional overrides so the connection can be pointed at the Synapse warehouse.
# Set to a role/warehouse that can read SYNAPSE_DATA_WAREHOUSE; leave None to use the connection's own.
ROLE = None
WAREHOUSE = None

PROJECT_ID = 2580853

if not CONNECTION_NAME:
    raise ValueError("Set CONNECTION_NAME to a connection from `snow connection list`.")

_connect_kwargs = {"connection_name": CONNECTION_NAME}
if ROLE:
    _connect_kwargs["role"] = ROLE
if WAREHOUSE:
    _connect_kwargs["warehouse"] = WAREHOUSE

# Silence the browser-auth banner ("Initiating login request ... Going to open: ...")
# that the connector prints during the OAuth/SSO handshake. The browser still opens.
with open(os.devnull, "w") as _devnull, \
        contextlib.redirect_stdout(_devnull), contextlib.redirect_stderr(_devnull):
    conn = snowflake.connector.connect(**_connect_kwargs)


def run_query(sql, params=None):
    """Execute SQL and return the result as a pandas DataFrame."""
    cur = conn.cursor()
    try:
        cur.execute(sql, params or {})
        columns = [col[0].lower() for col in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=columns)
    finally:
        cur.close()

## 1. Volume by billing source — latest version of files

Volume of data in STRIDES / AWS OD based on the **latest version** of each file (downloads default to the latest version). Reflects where files are stored at the time the query runs — it does not account for previous versions or deleted data.

In [ ]:
volume_latest_sql = """
SELECT
    CASE
        WHEN file_latest.bucket LIKE '%%opendata.sagebase.org' THEN 'AWS OD'
        ELSE 'STRIDES'
    END AS aws_billing,
    SUM(file_latest.content_size) / POWER(2, 40) AS size_in_tib,
    COUNT(*) AS number_of_files
FROM
    synapse_data_warehouse.synapse.file_latest
JOIN
    synapse_data_warehouse.synapse.node_latest
    ON file_latest.id = node_latest.file_handle_id
WHERE
    file_latest.status = 'AVAILABLE' AND
    file_latest.bucket != 'proddata.sagebase.org' AND
    file_latest.bucket is not NULL AND
    node_latest.project_id = %(project_id)s
GROUP BY
    aws_billing
ORDER BY
    size_in_tib DESC NULLS LAST
"""

volume_latest = run_query(volume_latest_sql, {"project_id": PROJECT_ID})
volume_latest

## 2. Volume by billing source and status — all versions

Entire volume of data including **all versions** of files across the project's lifecycle. Because data is usually not deleted from the S3 bucket, we use `synapse_event.node_event` and `file_latest` to inspect the state of every file handle ever associated with an ADKP file. This gives an accurate depiction of the data still within each bucket.

The availability status of the file referenced by the file handle.
* AVAILABLE: accessible via Synapse
* UNLINKED: The file or file version has been deleted in Synapse

In [ ]:
volume_all_versions_sql = """
WITH unique_filehandles AS (
    SELECT DISTINCT
        file_handle_id
    FROM
        synapse_data_warehouse.synapse_event.node_event
    WHERE
        project_id = %(project_id)s
)
SELECT
    CASE
        WHEN file_latest.bucket LIKE '%%opendata.sagebase.org' THEN 'AWS OD'
        ELSE 'STRIDES'
    END AS aws_billing,
    file_latest.status,
    SUM(file_latest.content_size) / POWER(2, 40) AS size_in_tib,
    COUNT(*) AS number_of_files
FROM
    synapse_data_warehouse.synapse.file_latest
JOIN
    unique_filehandles
    ON file_latest.id = unique_filehandles.file_handle_id
WHERE
    file_latest.bucket != 'proddata.sagebase.org' AND
    file_latest.bucket is not NULL
GROUP BY
    aws_billing, file_latest.status
ORDER BY
    size_in_tib DESC NULLS LAST
"""

volume_all_versions = run_query(volume_all_versions_sql, {"project_id": PROJECT_ID})
volume_all_versions

## Egress parameters

The egress reporting window.

In [ ]:
EGRESS_START = "2026-04-01"
# Any day in the final month; snapped to the last day of that month below.
# `EGRESS_END` is normalized to the last day of its month to match the original `LAST_DAY(...)` behavior.
EGRESS_END = (pd.Timestamp("2026-07-01") + pd.offsets.MonthEnd(0)).date().isoformat()

print(f"Egress window: {EGRESS_START} -> {EGRESS_END}")

## 3. Egress by billing source, month-over-month

Assesses egress month-over-month from migrating to AWS OD. Egress generated from the STRIDES bucket and AWS OD, across all versions of entities that exist in the project at query time.

NOTE: That these numbers will change based on when the query was run because we are querying for egress based on the files that exist at the time of query.

In [ ]:
egress_by_billing_sql = """
WITH adkp_downloads AS (
    SELECT
        objectdownload_event.user_id,
        objectdownload_event.record_date,
        objectdownload_event.file_handle_id,
        node_latest.annotations:annotations:study:value[0] AS study,
        file_latest.content_size,
        file_latest.bucket,
        CASE
            WHEN file_latest.bucket LIKE '%%opendata.sagebase.org' THEN 'AWS OD'
            ELSE 'STRIDES'
        END AS aws_billing
    FROM
        synapse_data_warehouse.synapse_event.objectdownload_event
    LEFT JOIN
        synapse_data_warehouse.synapse.node_latest
        ON objectdownload_event.association_object_id = node_latest.id
    LEFT JOIN
        synapse_data_warehouse.synapse.file_latest
        ON objectdownload_event.file_handle_id = file_latest.id
    WHERE
        objectdownload_event.project_id = %(project_id)s AND
        node_latest.project_id = %(project_id)s AND
        objectdownload_event.record_date BETWEEN %(start)s AND %(end)s AND
        file_latest.bucket != 'proddata.sagebase.org' AND
        file_latest.bucket is not NULL
)
SELECT
    aws_billing,
    DATE_TRUNC('MONTH', record_date) AS download_month,
    SUM(content_size) / POWER(2, 40) AS egress_in_tib
FROM
    adkp_downloads
GROUP BY
    aws_billing, download_month
HAVING
    aws_billing != 'None'
ORDER BY
    download_month DESC
"""

egress_by_billing = run_query(
    egress_by_billing_sql,
    {"project_id": PROJECT_ID, "start": EGRESS_START, "end": EGRESS_END},
)
egress_by_billing

In [ ]:
if not egress_by_billing.empty:
    pivot = (
        egress_by_billing
        .assign(download_month=lambda d: pd.to_datetime(d["download_month"]))
        .pivot_table(index="download_month", columns="aws_billing", values="egress_in_tib", aggfunc="sum")
        .sort_index()
    )
    ax = pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_title("ADKP egress by billing source, month-over-month")
    ax.set_xlabel("Download month")
    ax.set_ylabel("Egress (TiB)")
    ax.set_xticklabels([d.strftime("%Y-%m") for d in pivot.index], rotation=0)
    ax.legend(title="aws_billing")
    plt.tight_layout()
    plt.show()
else:
    print("No egress rows for the selected window.")

## 4. Egress by study

Egress from the STRIDES bucket, stack-ranked by study, for any study with more than 10 TiB egress per month. Egress on all versions of entities that exist in the project at query time.

Note: If an older version of a file is being downloaded and the file points to the file stored in the STRIDES bucket, you will continue to see egress even from datasets shifted to AWS OD unless those older verisons are deleted. Version deletions will have impact on provenance if people's scripts point to specific versions of files.

In [ ]:
egress_by_study_sql = """
WITH adkp_downloads AS (
    SELECT
        objectdownload_event.user_id,
        objectdownload_event.record_date,
        objectdownload_event.file_handle_id,
        node_latest.annotations:annotations:study:value[0] AS study,
        file_latest.content_size,
        file_latest.bucket,
        CASE
            WHEN file_latest.bucket LIKE '%%opendata.sagebase.org' THEN 'AWS OD'
            ELSE 'STRIDES'
        END AS aws_billing
    FROM
        synapse_data_warehouse.synapse_event.objectdownload_event
    LEFT JOIN
        synapse_data_warehouse.synapse.node_latest
        ON objectdownload_event.association_object_id = node_latest.id
    LEFT JOIN
        synapse_data_warehouse.synapse.file_latest
        ON objectdownload_event.file_handle_id = file_latest.id
    WHERE
        objectdownload_event.project_id = %(project_id)s AND
        node_latest.project_id = %(project_id)s AND
        objectdownload_event.record_date BETWEEN %(start)s AND %(end)s AND
        file_latest.bucket != 'proddata.sagebase.org' AND
        file_latest.bucket is not NULL
)
SELECT
    study,
    DATE_TRUNC('MONTH', record_date) AS download_month,
    SUM(content_size) / POWER(2, 40) AS egress_in_tib
FROM
    adkp_downloads
WHERE
    aws_billing != 'AWS OD'
GROUP BY
    study, download_month
HAVING
    egress_in_tib > 10
ORDER BY
    download_month DESC, egress_in_tib DESC
"""

egress_by_study = run_query(
    egress_by_study_sql,
    {"project_id": PROJECT_ID, "start": EGRESS_START, "end": EGRESS_END},
)
egress_by_study

In [ ]:
if not egress_by_study.empty:
    pivot = (
        egress_by_study
        .assign(download_month=lambda d: pd.to_datetime(d["download_month"]))
        .pivot_table(index="download_month", columns="study", values="egress_in_tib", aggfunc="sum")
        .sort_index()
    )
    ax = pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_title("ADKP STRIDES egress by study (>10 TiB/month)")
    ax.set_xlabel("Download month")
    ax.set_ylabel("Egress (TiB)")
    ax.set_xticklabels([d.strftime("%Y-%m") for d in pivot.index], rotation=0)
    ax.legend(title="study", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("No studies exceed 10 TiB/month egress from STRIDES for the selected window.")

## 5. Egress by study — STRIDES migration candidates

Studies to consider moving to AWS OD: egress from the STRIDES bucket, stack-ranked by study, for any study with more than 10 TiB egress per month. Egress on all versions of entities that exist in the project at query time but excludes all studies that have already been shifted to AWS OD: 'SEA-AD', 'AMP-AD_DiverseCohorts', 'ROSMAP', 'MSSM_PsychAD', 'MIT_ROSMAP_Multiomics', 'WGS_Harmonization', 'SEA-AD IAC'

In [ ]:
exclude_migrated_studies = []
egress_by_study_sql = """
WITH adkp_downloads AS (
    SELECT
        objectdownload_event.user_id,
        objectdownload_event.record_date,
        objectdownload_event.file_handle_id,
        node_latest.annotations:annotations:study:value[0] AS study,
        file_latest.content_size,
        file_latest.bucket,
        CASE
            WHEN file_latest.bucket LIKE '%%opendata.sagebase.org' THEN 'AWS OD'
            ELSE 'STRIDES'
        END AS aws_billing
    FROM
        synapse_data_warehouse.synapse_event.objectdownload_event
    LEFT JOIN
        synapse_data_warehouse.synapse.node_latest
        ON objectdownload_event.association_object_id = node_latest.id
    LEFT JOIN
        synapse_data_warehouse.synapse.file_latest
        ON objectdownload_event.file_handle_id = file_latest.id
    WHERE
        objectdownload_event.project_id = %(project_id)s AND
        node_latest.project_id = %(project_id)s AND
        objectdownload_event.record_date BETWEEN %(start)s AND %(end)s AND
        file_latest.bucket != 'proddata.sagebase.org' AND
        file_latest.bucket is not NULL AND
        study not in ('SEA-AD', 'AMP-AD_DiverseCohorts', 'ROSMAP', 'MSSM_PsychAD', 'MIT_ROSMAP_Multiomics', 'WGS_Harmonization', 'SEA-AD IAC')
)
SELECT
    study,
    DATE_TRUNC('MONTH', record_date) AS download_month,
    SUM(content_size) / POWER(2, 40) AS egress_in_tib
FROM
    adkp_downloads
WHERE
    aws_billing != 'AWS OD'
GROUP BY
    study, download_month
HAVING
    egress_in_tib > 10
ORDER BY
    download_month DESC, egress_in_tib DESC
"""

egress_by_study = run_query(
    egress_by_study_sql,
    {"project_id": PROJECT_ID, "start": EGRESS_START, "end": EGRESS_END},
)
egress_by_study

In [ ]:
if not egress_by_study.empty:
    pivot = (
        egress_by_study
        .assign(download_month=lambda d: pd.to_datetime(d["download_month"]))
        .pivot_table(index="download_month", columns="study", values="egress_in_tib", aggfunc="sum")
        .sort_index()
    )
    ax = pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_title("ADKP STRIDES egress by study (>10 TiB/month)")
    ax.set_xlabel("Download month")
    ax.set_ylabel("Egress (TiB)")
    ax.set_xticklabels([d.strftime("%Y-%m") for d in pivot.index], rotation=0)
    ax.legend(title="study", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("No studies exceed 10 TiB/month egress from STRIDES for the selected window.")